**Step 1: Install Necessary Libraries**



The following cell installs essential libraries for fine-tuning.

* **Unsloth:** A framework for efficient model management and fine-tuning.

* **Xformers:** Optimizes attention mechanisms for handling large sequences.

* **TRL (Transformers Reinforcement Learning):** Provides tools for reinforcement learning-based model tuning.

* **PEFT (Parameter Efficient Fine-Tuning):** Reduces memory requirements during fine-tuning by updating only a subset of model parameters.

* **BitsAndBytes:** Enables efficient quantization techniques, such as 4-bit precision.




Using these libraries ensures our environment is equipped for both memory-efficient and effective LLM fine-tuning.

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

**Step 2: Load Pretrained Model and Tokenizer**


**What is Unsloth and What Is It Used For?**

Unsloth is a streamlined framework designed to simplify and optimize the process of working with large language models (LLMs). Think of it as your ultimate toolkit for fine-tuning and deploying LLMs with efficiency and ease.

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048
dtype = None
load_in_4bit = True

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


**Why Did We Load the Model from Unsloth?**

We loaded the model from Unsloth because it provides a pre-configured, optimized environment tailored for efficient LLM fine-tuning and deployment. Here’s why this choice makes sense:

* **Memory Efficiency:** Unsloth’s models are pre-quantized, often using techniques like 4-bit precision, which significantly reduces memory requirements without compromising performance.
* **Long-Context Support:** The framework incorporates advanced features like RoPE (Rotary Position Embedding) scaling, making it ideal for tasks requiring long input sequences.
* **Fine-Tuning Ready:** Models from Unsloth are designed with parameter-efficient techniques in mind, ensuring smooth integration with LoRA and QLoRA.
* **Ease of Use:** By handling complex setups internally, Unsloth eliminates the need for extensive manual configurations, saving time and reducing errors.

The model loaded here is "unsloth/llama-3-8b-bnb-4bit," a lightweight yet powerful variant for tasks requiring large language models.

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

==((====))==  Unsloth 2025.8.10: Fast Llama patching. Transformers: 4.55.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

**Step 3: Apply Parameter-Efficient Fine-Tuning (LoRA)**



Parameter-Efficient Fine-Tuning (PEFT) is a more efficient form of instruction-based fine-tuning. Full LLM fine-tuning is resource-intensive, demanding considerable computational power, memory, and storage. PEFT addresses this by updating only a select set of parameters while keeping the rest frozen. This reduces the memory load during training and prevents the model from forgetting previously learned information. PEFT is particularly useful when fine-tuning for multiple tasks. Among the common techniques to achieve PEFT, LoRA and QLoRA are widely recognized for their effectiveness.

**What is LoRA?**

LoRA (Low-Rank Adaptation) is a method that fine-tunes two smaller matrices instead of the entire weight matrix of a pre-trained LLM. These smaller matrices form a LoRA adapter, which is then applied to the original LLM. The fine-tuned adapter is much smaller in size compared to the original model, often only a small percentage of its size. During inference, this LoRA adapter is combined with the original LLM.

In our notebook, LoRA adapters are applied:

* **Target Modules:** Defines which parts of the model are fine-tuned, like query, key, and value projections.
* **LoRA Alpha & Dropout:** Control the adaptation strength and regularization.
* **Gradient Checkpointing:** Reduces memory usage during training by recomputing intermediate states.
* **Random State:** Ensures reproducibility.

This step ensure that wwe are only modifying specific parameters (around 10% of all parameters).

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth 2025.8.10 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [ ]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096, padding_idx=128255)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor


**Step 4: Load and Preprocess the Dataset for Fine-Tuning**

**What is an LLM Dataset?**

An LLM dataset is a collection of text data used for training and fine-tuning language models. These datasets contain various types of text, such as questions, answers, documents, or dialogues, and are tailored for specific tasks or domains. The quality of the dataset significantly influences the model's performance and accuracy.


When we want to fine-tune a model for a specific use case, we can create a custom dataset that falls into any of the above categories. By curating a dataset that is tailored to the task at hand, we can optimize the model's performance to meet the specific needs of the application. This custom dataset allows us to focus on the relevant information and ensure that the model is fine-tuned with the most appropriate data for the target use case, leading to more accurate and effective results.








In [ ]:
chat_prompt = """
### Instruction:
{}

### Input:
{}

### Response:
{}"""

This function ensures the input data aligns with the model’s requirements, reducing inconsistencies during training.

In [ ]:
EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN
def formatting_prompts_func(examples):
    instruction = ""
    inputs       = examples["input"]
    outputs      = examples["response"]
    texts = []
    for input, output in zip(inputs, outputs):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = chat_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }
pass

In [ ]:
from datasets import load_dataset

dataset = load_dataset("bongpheng/credit_risk_ds_100k", split = "train")
dataset = dataset.map(formatting_prompts_func, batched = True,)

Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

In [ ]:
dataset

Dataset({
    features: ['input', 'response', 'risk_classification', 'text'],
    num_rows: 100000
})

In [ ]:
import pprint
#Here are a few examples of what the data looks like
pprint.pprint(dataset[250])
pprint.pprint(dataset[260])
pprint.pprint(dataset[270])

{'input': 'Applicant with monthly income $9778.49, debt obligations $11262.84, '
          'credit score 350, and 2 late payments in the past year.',
 'response': 'Given an income of $9778.49, a debt of $11262.84 resulting in a '
             'DTI of 1.15, and 2 late payments with a credit score of 350, the '
             "applicant is considered 'Very High Risk'.",
 'risk_classification': 'Very High Risk',
 'text': '\n'
         '### Instruction:\n'
         '\n'
         '\n'
         '### Input:\n'
         'Applicant with monthly income $9778.49, debt obligations $11262.84, '
         'credit score 350, and 2 late payments in the past year.\n'
         '\n'
         '### Response:\n'
         'Given an income of $9778.49, a debt of $11262.84 resulting in a DTI '
         'of 1.15, and 2 late payments with a credit score of 350, the '
         "applicant is considered 'Very High Risk'.<|end_of_text|>"}
{'input': 'Applicant with monthly income $8156.03, debt obligations $3447.75, '
 

**Step 5: Configure the Training Parameters**

Now let's define the training parameters, including the model, tokenizer, and dataset, along with key training settings like batch size, gradient accumulation steps, learning rate, and maximum training steps. The TrainingArguments specify additional configurations, such as optimization with the AdamW optimizer, weight decay, and logging frequency.

* **Learning Rate:** Controls the speed at which the model updates during training.

* **Batch Size:** The number of samples processed in one iteration.

* **Epochs:** The number of times the model passes through the entire training dataset.

* **Logging Directory:** Specifies where to store training logs, useful for monitoring progress.

The Trainer simplifies the configuration and management of the fine-tuning process, ensuring a balanced and efficient setup. We did 60 steps to make finetuning faster, **but you can set num_train_epochs=1 for a full run, and turn off max_steps=None**.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = True,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/100000 [00:00<?, ? examples/s]

**Step 6: Start Training**

The line `trainer_stats = trainer.train() `initiates the fine-tuning process using the SFTTrainer. It triggers the training loop, where the model learns from the provided dataset based on the configurations defined earlier. we can see the loss is decreasing during training, this means the model is learning and improving its performance. In machine learning, loss represents how well the model's predictions match the actual target values. When the loss decreases over time, it indicates that the model is gradually adjusting its parameters to make more accurate predictions.

In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100,000 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ubiaiinc (ubiaiinc-esprit) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss,entropy
1,2.302400,0
2,2.286600,No Log
3,2.246600,No Log
4,2.039400,No Log
5,1.737600,No Log
6,1.397000,No Log
7,0.921000,No Log
8,0.638300,No Log
9,0.476400,No Log
10,0.437900,No Log


**Step 7: Perform Inference with the Fine-Tuned Model to Evaluate output**

When we generate text with the fine-tuned model, the output typically includes the full structure of the input prompt along with the model's response. For instance, the output for this example looks like this:

In [ ]:
FastLanguageModel.for_inference(model)

inputs = tokenizer(
    [
        chat_prompt.format(
            "",  # instruction
            "Applicant with monthly income $1137.38, debt obligations $766.31, credit score 600, and 1 late payments in the past year.",  # input
            "",  # output
        )
    ],
    return_tensors="pt"
).to("cuda")

outputs = model.generate(**inputs, max_new_tokens=64, use_cache=True)
decoded_output = tokenizer.batch_decode(outputs)[0]  # Decode the output

# Extracting the response part
response = decoded_output.split("### Response:")[-1].strip()  # Get text after "### Response:"
response = response.split("<|end_of_text|>")[0].strip()  # Remove the end token if present

print(response)


Given an income of $1137.38, a debt of $766.31 resulting in a DTI of 0.67, and 1 late payments with a credit score of 600, the applicant is considered 'High Risk'.


**Comparing response to dataset answer:**

As you can see, when we asked the same question twice, the model gave the same answer but written in different ways. This shows that the model is relying on the dataset while also being able to generalize.

**Step 8: save and load**

In [ ]:
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")
# model.push_to_hub("your_name/lora_model", token = "...") # If you want to save online
# tokenizer.push_to_hub("your_name/lora_model", token = "...") # If you want to save online

('lora_model/tokenizer_config.json',
 'lora_model/special_tokens_map.json',
 'lora_model/tokenizer.json')

In [ ]:
#Let's zip our model folder
import shutil
import os
folder_path = "/content/lora_model"
zip_file_path = "/content/lora_model.zip"

shutil.make_archive(zip_file_path.replace(".zip", ""), 'zip', folder_path)

'/content/lora_model.zip'

In [ ]:
#Now if we want to use our model again we can just load it:

if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "lora_model", #model folder
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model)

In [ ]:
chat_prompt = """
### Instruction:
{}

### Input:
{}

### Response:
{}"""

In [ ]:
inputs = tokenizer(
    [
        chat_prompt.format(
            "",  # instruction
            "Applicant with monthly income $1137.38, debt obligations $766.31, credit score 600, and 1 late payments in the past year.",  # input
            "",  # output
        )
    ],
    return_tensors="pt"
).to("cuda")

outputs = model.generate(**inputs, max_new_tokens=64, use_cache=True)
decoded_output = tokenizer.batch_decode(outputs)[0]

response = decoded_output.split("### Response:")[-1].strip()
response = response.split("<|end_of_text|>")[0].strip()

print(response)

Given an income of $1137.38, a debt of $766.31 resulting in a DTI of 0.68, and 1 late payments with a credit score of 600, the applicant is considered 'High Risk'.


And that's a wrap! Keep exploring, keep learning, and enjoy the process.